# Consolidation Exercise 1 — Autonomous Robot Navigation

So far, we've covered:

+ Variables, data types and operators
+ Functions
+ If-then-else
+ Loops (for, while)

This week, we'll be doing our very first *consolidation* exercise.
In prior weeks, we've been looking at specific elements of Python syntax. 
This is necessary, but has the drawback that when you're solving the exercises, you're given a big clue about what you should do — 
_eg_ in Week 4, when we looked at loops, there's a fairly good chance that you'll need to use loops to solve the exercises. 

The purpose of consolidation exercises is to give you practice writing code in a scenario that is closer to what you'll face after this course, where it won't always be immediately obvious how a piece of code should be written. 
This practice will be invaluable after the course when you are programming *in the wild*, but at first you may find it hard to get started — given such a big problem, what should you do first? 
My advice is to start by simplifying the problem and building your solution up bit-by-bit. 
As the old adage goes, the only way to eat an elephant is one bite at a time!

Consolidation exercises are also a chance for you to write slightly longer pieces of code than we'll generally have time to do in the lab. 
This means you'll need to think much more about how to structure and organise your code.

You have hopefully also noticed that there's no pre-learning materials or comprehension checks for this week; the flip-side of that is that we expect you to spend slightly more time (a few hours) working on these problems that you would a regular lab sheet. 
Consolidation exercises are also much closer to what we would expect you to complete for your individual programming project (**worth 70% of your mark** in this course) and so represent the best chance for you to "practice" the coursework before you complete it. 
As such, we'll be providing formative feedback for consolidation exercises submitted on Blackboard.

## Overview

We've been building a piece of code for simulating the movement of a two-wheeled robot.

<div style="text-align:center;">
    <img src="https://github.com/engmaths/SEMT10002_2025/blob/main/media/week_1/romi.jpg?raw=true" height="250ex"/>
</div>

So far you've written some code for storing information, for calculating speeds and turning rates,for detecting collisions and for integrating movement. We've organised this for you into the sort of structure you can expect to see for larger pieces of Python code and created some scenarios for you to explore. We've also provided a simple plotting library we can use to visualise the movement of the robot. Today, we'll try to add:

1. Basic goal-finding capabilities.
2. Insect-inspired navigation.


## Preliminaries

We've added a plotting library you can use to visualise the movement of the robot. It can plot the robot's current (red triangle) and previous (black triangles) positions, the arena walls, goal position, and obstacle (if one exists). 

The plotting library code is contained in the file "robot_plotter.py". Feel free to take a look at this, but understanding how the code work is not necessary for this exercise. We'll have a deeper look at libraries and plotting later in the year. For now, what you need to know is that the library gives us three functions:

1. ```init_plot(x, y, heading)```

This function is used to initialise the plot. You need to call it once at the start of a scenario.

2. ```snapshot(x, y, heading)```

This function is used to record the position of the robot. The position provided as input will be the "current position". Previous positions are saved so we can display the path.

3. ```show_plot(x_min, x_max, y_min, y_max, goal=None, obstacle=None, pause=0)```

This function draws the plot to the screen -- if you don't call it, nothing will be shown! Note particularly the pause parameter -- this determines how long a plot will display for. It defaults to 0, which the plotting library will interpet as "show forever". If you call show_plot with a pause of 0, your code will stop running until you close the plot window. If you call show_plot with a pause of e.g. 0.1, then the plot will be shown for 0.1 seconds before the code continues. This lets us call show_plot repeatedly within a loop to generate an animation of the robot moving around.


Secondly, we've restructured the file slightly from what we had in Exercise 4. We've structured the code as is conventional for Python files, following the high-level structure of:

1. Imports
2. Constants
3. Function definitions
4. Main Function
5. Call to main function

Let's briefly go through the code, so we can understand what it does.

We start with imports -- we need to import the functions from our robot library, some mathematical functions and a random number generator.

In [ ]:
import math
import random
from robot_plotter import init_plot, snapshot, show_plot

Next, we have some variable definitions. You'll notice that we've used UPPER CASE letters for the variable names here. That's to indicate that these variable are constants and **should not** be changed later in the code. Here, we're using constants to store the values associated with the robot's physical parameters, and the shape of the arena the robot is confined within. 

In [ ]:
WORLD_X_MIN = 0.0
WORLD_X_MAX = 10000.0
WORLD_Y_MIN = 0.0
WORLD_Y_MAX = 10000.0
GOAL_TOLERANCE = 5.0

WHEEL_SEPARATION_IN= 5.906   # distance between the two wheels
WHEEL_RADIUS_IN     = 1.378   # radius of each wheel
ROBOT_RADIUS_IN     = 3.248   # radius of the robot body
MAX_WHEEL_SPEED = 10

WHEEL_SEPARATION_MM = WHEEL_SEPARATION_IN * 25.4
WHEEL_RADIUS_MM = WHEEL_RADIUS_IN * 25.4
ROBOT_RADIUS_MM = ROBOT_RADIUS_IN * 25.4

Next, we have our function definitions -- not shown in full, but they run from line 22 to line 194. The top 8 functions are from previous exercises, so we won't cover them in detail here. Essentially, they can be used to calculate the robot's new position if we are given a time and some angular wheel speeds. Our first new function is ```run_scenario```

In [ ]:
def run_scenario(controller, start_x, start_y, start_theta, max_time = 50, goal_x=None, goal_y=None, obstacle_x=None, obstacle_y=None, obstacle_radius=None):

    #Set initial parameters
    x = start_x
    y = start_y
    theta = start_theta
    t = 0
    step = 1

    # Check to see if we have a goal or not.
    if goal_x and goal_y:
        # If we do, make it into a tuple we can pass to the plotting function
        goal = (goal_x, goal_y)
    else:
        goal = None

    # Check to see if we have an obstacle or not
    if obstacle_x and obstacle_y and obstacle_radius:
        #If we do, make it into a tuple we can pass to the plotting function
        obstacle = (obstacle_x, obstacle_y, obstacle_radius)
    else:
        obstacle = None

    #Initialise the plot
    init_plot(x, y, theta)
    #Show the initial configuration for 0.1 seconds
    show_plot(WORLD_X_MIN, WORLD_X_MAX, WORLD_Y_MIN, WORLD_Y_MAX, goal, obstacle, pause=0.1)

    #Main simulation loop
    while t < max_time:
        #Ask the controller what speeds we should set the robot wheels to.
        left_speed, right_speed = controller(x, y, theta, goal_x, goal_y, obstacle_x, obstacle_y, obstacle_radius)
        #Make sure we don't exceed the maximum allowable speeds.
        left_speed = max(-MAX_WHEEL_SPEED, min(MAX_WHEEL_SPEED, left_speed))
        right_speed = max(-MAX_WHEEL_SPEED, min(MAX_WHEEL_SPEED, right_speed))
        #Integrate for one second
        x, y, theta = get_new_position(step, left_speed, right_speed, x, y, theta)
        #Draw the updated position
        snapshot(x, y, theta)
        show_plot(WORLD_X_MIN, WORLD_X_MAX, WORLD_Y_MIN, WORLD_Y_MAX, goal, obstacle, pause=0.1)
        t = t + step

        #Check we're still inside the arena
        if not is_inside_arena(x, y):
            print("Error - robot tried to leave the arena")
            break 
        #Check we haven't collided with the obstacle
        if obstacle and has_collided(x, y, obstacle_x, obstacle_y, obstacle_radius):
            print("Error - robot collised with the obstacle")
            break

    #Now the main loop has finished, let's see how close we are to the goal.
    if goal:
        x_error = abs(x - goal_x)
        y_error = abs(y - goal_y)
        total_error = math.sqrt(x_error**2 + y_error**2)
        print("Robot has stopped moving -- distance to goal is:", total_error)
        if total_error < GOAL_TOLERANCE:
            print ("You reached the goal -- well done!")

    # Plot the map one final time, with a longer (5 seconds) pause so we can inspect it.
    show_plot(WORLD_X_MIN, WORLD_X_MAX, WORLD_Y_MIN, WORLD_Y_MAX, goal, obstacle, pause=5.0)

This function may seem long, but it's actually pretty simple. We call it with some inputs -- the start position, max running time, goal position and obstacle details. The first part of the code (lines 105-124) just re-package these inputs into the form we'll use later. We then initialise our plot on lines 126-129. Line numbers refer to the file "robot.py" -- they'll be different in the notebook, so check the robot.py file if your not sure which section we're referring to.

We then enter the main simulation loop. This loop will ask a *controller* what the wheel speeds should be. For now, controller just returns random values. Your task in this assignment is to change the code in the controller function so that the robot completes a series of tasks. More on that later though! 

Once the controller has provided wheel speeds, we first check that these don't exceed our max speeds (line 136-137) and then use our function defined above to integrate the movement for 1 second. This gives us a new position and heading. We then plot the robot's new position (lines 141-142), check the robot hasn't left the arena (lines 145-148), check for collisions (lines 150-152). This loop continues until we've reached a total time equal to max_time.

Once the main loop has finished, we check to see if we reached the goal or not (lines 154-161) and then show a final plot (lines 164). 

Next, we have four *controller* functions. 

In [ ]:
def controller_1(x, y, theta, goal_x, goal_y, obstacle_x=None, obstacle_y=None, obstacle_radius=None):

    left = random.random()
    right = random.random()

    return left, right 

def controller_2(x, y, theta, goal_x, goal_y, obstacle_x=None, obstacle_y=None, obstacle_radius=None):

    left = random.random()
    right = random.random()

    return left, right 

def controller_3(x, y, theta, goal_x, goal_y, obstacle_x=None, obstacle_y=None, obstacle_radius=None):

    left = random.random()
    right = random.random()

    return left, right 

def controller_4(x, y, theta, goal_x, goal_y, obstacle_x=None, obstacle_y=None, obstacle_radius=None):

    left = random.random()
    right = random.random()

    return left, right 

Each controller corresponds to a particular task (introduced later). Your job is to write the code for each controller so that it can complete the corresponding task. The controller is provided some information about the current state of the robot and its environment and must return two values -- the appropriate angular wheel speeds for the next movement step. 

Next, we have the *main* function. 

In [ ]:
def main():

    #Use this function to determine which scenario to run. 

    # Scenario 1 -- Can you move in a straight line and stop at the goal?
    start_x = 5000
    start_y = 1000
    start_theta = math.pi / 2
    goal_x = 5000
    goal_y = 8000
    max_time = 100
    run_scenario(controller_1, start_x, start_y, start_theta, max_time, goal_x, goal_y)

    # Scenario 2 -- Can you turn and then move in a straight line to the goal?
    '''
    start_x = 5000
    start_y = 1000
    start_theta = math.pi
    goal_x = 5000
    goal_y = 8000
    max_time = 100
    run_scenario(controller_2, start_x, start_y, start_theta, max_time, goal_x, goal_y)
    '''

    # Scenario 3 -- Can you naivgate past a fixed obstacle?
    '''
    start_x = 5000
    start_y = 1000
    start_theta = math.pi
    goal_x = 5000
    goal_y = 8000
    obstacle_x = 5000
    obstacle_y = 5000
    obstacle_radius = 500
    max_time = 100
    run_scenario(controller_3, start_x, start_y, start_theta, max_time, goal_x, goal_y, obstacle_x, obstacle_y, obstacle_radius)
    '''

    # Scenario 4 -- Can you navigate past a randomly located obstacle?
    '''
    start_x = 100 + 4900 * random.random()
    start_y = 100 + 900 * random.random()
    start_theta = math.pi
    goal_x = 5000 + (4000 * random.random())
    goal_y = 6000 + (3000 * random.random())
    obstacle_x = 2500 + (5000 * random.random())
    obstacle_y = 2500 + (5000 * random.random())
    obstacle_radius = 2000    

    while (has_collided(start_x, start_y, obstacle_x, obstacle_y, obstacle_radius) or has_collided(goal_x, goal_y, obstacle_x, obstacle_y, obstacle_radius)):
       obstacle_x = 2500 + (5000 * random.random())
       obstacle_y = 2500 + (5000 * random.random())
    
    max_time = 200
    run_scenario(controller_4, start_x, start_y, start_theta, max_time, goal_x, goal_y, obstacle_x, obstacle_y, obstacle_radius)
    '''

The main function is used to determine which scenario is run. Each scenario consists of some parameters and a call to the "run_scenario" function. To begin with, we've commented out all but the first scenario. We suggest you work on these in sequence -- so solve task 1, then comment out the code for running task 1, and comment in the code for running task 2. Then try to solve task 2 and so on.

Finally, we have some code that calls the main function. 

In [ ]:
if __name__ == "__main__":
    main()

We've put this inside an "if statement". This might look a bit funny, but is convention for Python code. The reason is that a particular Python file could be run directly *or* imported into another bit of Python code. In general, we want main to run *only* when the file is run directly and not when it is imported. The *if* statement prevents main from running if the code is imported. It's not strictly needed here, but is a convention for Python code, so we may as well get used to seeing it now. Don't worry too much if you don't fully understand this! We'll cover libraries in more detail later in the course. 

Hopefully, you can see how this code follows the convention for the structure of a Python file:

1. Imports
2. Constants
3. Function definitions
4. Main Function
5. Call to main function

In general, you should expect to see us use this structure for larger code examples we give you throughout the course. We'll also expect you to follow it yourselves when writing your own code. 

## Task(s)

### Task 1: Moving in a straight-line towards the target. 

Scenario 1 starts with the robot facing the target. You just need to edit the controller (i.e. the function ```controller_1```) to drive straight until you reach the target. You could do this by calculating how far away the goal is and then determining how much time you need to turn the wheels at a set speed to reach the target. This is similar to what we did last week for the "drive in a square" part of task 4 and is known as an "open loop" controller. A better approach would be to implement a *feedback* controller. In a feedback controller, we calculate an error and use the error sign and magnitude to determine an appropriate control input.

It may seem like the distance to the goal is a good "error" signal for this task. Unfortunately, a distance only has a magnitude (and not a sign). If we use distance to the goal as an error, it'll always be positive and the robot will never reverse even if it goes past the goal. 

Instead, we use the distance to the goal *along the direction the robot is facing*:

$$
\text{error} = (g_x - x)\cos\theta + (g_y - y)\sin\theta
$$

where $x$, $y$ and $\theta$ are the robot's position and heading, and $g_x$, $g_y$ is the goal position. This is positive if the goal is ahead of the robot, and negative if it is behind.

We then set both wheel speeds proportional to the error:

$$
\text{wheel speed} = k \times \text{error}
$$

where $k$ is a constant. Try a few different values. If $k$ is too large, the robot will overshoot the goal and oscillate back and forth (or worse!). If it's too small, it will take a long time to arrive. 


### Task 2: Turning and then movingin a straight line towards the target

Scenario 2 starts with the robot facing in the wrong direct. We suggest you write a solution (by editting the function ```controller_2```) that first turns the robot to face the target and then drive straight until it reaches it. Can you re-use your approach from task 1 to do this? Remember to comment out the code for running scenario 1 in the function ```main``` and uncomment the code for running scenario 2. To do this task, you'll probably need to use some trigonometry to calculate the heading to the goal. The math library's ```atan2(y, x)``` function for calculating the inverse tangent will be helpful here -- but notice the odd calling convention (y first, then x second). 

### Task 3: Avoiding a fixed obstacle

Scenario 3 starts the same as Task 2, but we've now added a small obstacle which blocks the direct path from the start position to the goal. Can you write some code to navigate around the obstacle? A simple approach would be move horizontally until you are past the left (or right) hand side of the obstacle. Then move vertically until you are past the top of the obstacle. Then finally, navigate to the goal position. Try to do this with the feedback controllers you created for earlier tasks.

### Task 4: Avoiding a random obstacle with the bug 0 algorithm.

Scenario 4 is the same as Task 3, but now the positions (start, goal and obstacle) are randomised *and* we would like you to solve it by implementing a *bug* algorithm.

## Bug Algorithms

For all three tasks above, our controller has (probably) used the information about the obstacle to determine how to act. In practice, often in robotics, we don't know about obstacles until we've (almost) hit them. Sensors can tell us e.g. there is something ahead of us, but not what it is, or what shape it is. This means we can't always determine a safe waypoint that we can navigate to. One way of overcoming this is with **bug** algorithms, which are insect inspired autonomous navigation algorithms. We assume the robot can measure the distance and direction towards the goal, but otherwise only has sensors for detecting collisions. To respect this assumption in our code, we have to *only* use information about the obstacle (i.e. the variables obstacle_x, obstacle_y and obstacle_radius) as inputs to the function has_collided. We can use the information returned by the has_collided function - but not the obstacle information directly. 

### Bug 0

<img src="https://raw.githubusercontent.com/engmaths/SEMT10002_2025/refs/heads/main/media/week_5/bug0.png" width="21%" style="display:block; margin:0 auto;"/>

The simplest bug algorithm is **Bug 0**, which is as follows:

1. Head towards goal until you reach the goal or hit an obstacle.
2. If you hit an obstacle, move around it until you can head towards the goal again.
3. Go to step 1.

####  Implementing Bug 0 

To implement bug 0, you'll need:

1. Some code for determining when you are about to hit an obstacle. Remember, you can re-use existing functions for this -- there's no reason controller can't call ```get_new_position``` or ```has_collided```.
2. Some code for determining where to move if you are about to hit an obstacle. In bug 0, the idea is to always turn left: starting from the direction of the goal, try headings further and further to the left (e.g. in steps of 10°) until you find one where a step forward would *not* collide. That direction will run roughly along the edge of the obstacle

**Reminder for Task 4 (and 5):** a bug algorithm assumes the robot can't see the obstacle, only sense when it's about to hit it. So your controller may only use `obstacle_x`, `obstacle_y` and `obstacle_radius` by passing them to `has_collided`.


### Better Bugs

**Bug 0**, while simple, can not solve all maps. 
There are two modifications of **Bug 0** (known respectively as **Bug 1** and **Bug 2**). 

The **Bug 1** algorithm is as follows:

1. Head towards goal.
2. If an obstacle is encountered, circumnavigate it *and* remember which point on the perimeter is closest to the goal.
3. Return to the closest point and then continue towards the goal.

<img src="https://raw.githubusercontent.com/engmaths/SEMT10002_2025/refs/heads/main/media/week_5/bug1.png" width="20%" style="display:block; margin:0 auto;"/>

**Bug 1** will solve more scenarios than **Bug 0**, but often produces very long paths (due to first circumnavigating each obstacle). 
**Bug 2** attempts to solve this by introducing the concept of the **m-line**, the shortest line from start position to goal position (ignoring obstacles). 
**Bug 2** is then:

1. Head towards goal on the **m-line** (our goal-seeking behaviour from above will do this for you!).
2. If an obstacle is in the way, follow it until you encounter the **m-line** closer to the goal than when you left it.
3. Leave obstacle and continue towards the goal.

<img src="https://raw.githubusercontent.com/engmaths/SEMT10002_2025/refs/heads/main/media/week_5/bug2.png" width="20%" style="display:block; margin:0 auto;"/><br/>

### Task 5 - Implementing a better bug.

Implement one or more  of the better bug algorithmns and use them to solve Scenario 4.

Bug 0 can decide what to do using only where the robot is *right now*. Bug 1 and Bug 2 can't: they need to remember things between steps, such as where the robot first hit the obstacle, or the closest point to the goal it has seen so far.

Unfortunatley, as written, our controllers have no memory. Every call starts with fresh local variables, which disappear when the function returns (remember the scope seminar). There are two ways to fix this:

1. **Global variables**, updated with the `global` keyword. This works, but we saw in the seminar that it makes code hard to debug and reuse. Please don't.
2. **Pass the memory in and return it back out.** Your controller takes its memory as extra inputs, and returns the updated memory alongside the wheel speeds. `run_scenario` stores what was returned, and hands it back on the next call. This is the same pattern as `number = square(number)`.

For option 2, you'll need to modify `run_scenario`:
- Before the main loop, create your memory variables with sensible starting values.
- Inside the loop, change the controller call so it passes the memory in and receives the updated memory back.

Changing `run_scenario` will affect the other controllers too. Decide how you'll deal with that.

